# MitigLLM - Clean SFT QLoRA Pipeline

This notebook is the clean training entry point for MitigLLM.

Goal: fine-tune `Mistral-7B-Instruct-v0.3` for cybersecurity mitigation generation using a corrected SFT dataset.

Recommended order:

1. Check GPU and paths
2. Build the enhanced dataset
3. Inspect quality report and samples
4. Train SFT QLoRA
5. Test the trained adapter
6. Use the adapter in the Django backend

In [ ]:
from pathlib import Path
import json
import os
import torch

PROJECT_DIR = Path(r"C:/Users/user/Desktop/MitigLLM")
BASE_MODEL = Path(r"C:/Users/user/Desktop/models/Mistral-7B-Instruct-v0.3")
RAW_DATASET = PROJECT_DIR / "final_fully_cleaned_and_aligned.json"
CURATED_DIR = PROJECT_DIR / "data" / "curated"
OUTPUT_DIR = PROJECT_DIR / "models" / "mitigllm-mistral-sft"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Project:", PROJECT_DIR)
print("Base model exists:", BASE_MODEL.exists(), BASE_MODEL)
print("Dataset exists:", RAW_DATASET.exists(), RAW_DATASET)

## 1. Install Dependencies

Run this once in a fresh environment. If you already installed the packages, skip this cell.

For Windows, `bitsandbytes` support depends on your CUDA/PyTorch setup. If 4-bit loading fails locally, use Colab/Kaggle or WSL with CUDA.

In [ ]:
# Uncomment if needed
# !pip install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece safetensors huggingface_hub

## 2. Build Curated Dataset

This creates:

- `data/curated/train.jsonl`
- `data/curated/val.jsonl`
- `data/curated/test.jsonl`
- `data/curated/eval_prompts.jsonl`
- `data/curated/report.json`

The script keeps good source examples and repairs weak/short mitigations with structured Purple Team mitigation guidance.

In [ ]:
%cd C:/Users/user/Desktop/MitigLLM
!python training_pipeline/build_curated_dataset.py --input final_fully_cleaned_and_aligned.json --output-dir data/curated

## 3. Inspect Dataset Quality

In [ ]:
report = json.loads((CURATED_DIR / "report.json").read_text(encoding="utf-8"))
print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
import itertools

with (CURATED_DIR / "train.jsonl").open(encoding="utf-8") as f:
    for line in itertools.islice(f, 3):
        row = json.loads(line)
        print("QUALITY:", row["quality"])
        print("INPUT:", row["input"][:500])
        print("OUTPUT:", row["output"][:700])
        print("-" * 100)

## 4. Train SFT QLoRA

Start with conservative settings. Increase epochs only after checking validation behavior.

If your GPU has limited VRAM, keep:

- `--batch-size 1`
- `--grad-accum 8`
- `--max-seq-length 1024`
- `--use-4bit`
- `--gradient-checkpointing`

In [ ]:
%cd C:/Users/user/Desktop/MitigLLM
!python training_pipeline/train_sft_qlora.py ^
  --model C:/Users/user/Desktop/models/Mistral-7B-Instruct-v0.3 ^
  --train-file data/curated/train.jsonl ^
  --val-file data/curated/val.jsonl ^
  --output-dir models/mitigllm-mistral-sft ^
  --max-seq-length 1024 ^
  --epochs 2 ^
  --batch-size 1 ^
  --grad-accum 8 ^
  --lr 2e-4 ^
  --use-4bit ^
  --gradient-checkpointing

## 5. Quick Inference Test

This loads the base model and the trained LoRA adapter. Use it after training finishes.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL), trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL),
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, str(OUTPUT_DIR))
model.eval()

prompt = """<s>[INST] You are MitigLLM, a Purple Team cybersecurity assistant. Given a vulnerability description, produce practical mitigation guidance. Prioritize patching, configuration changes, compensating controls, detection, and validation.

Vulnerability description:
A web application is vulnerable to SQL injection in the login form because user input is concatenated directly into a SQL query. [/INST]"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=220, temperature=0.4, top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0], skip_special_tokens=True))

## 6. Configure Django Backend

After training, point the backend to the base model and the LoRA adapter:

```powershell
$env:MITIGLLM_BASE_MODEL_PATH="C:\Users\user\Desktop\models\Mistral-7B-Instruct-v0.3"
$env:MITIGLLM_ADAPTER_PATH="C:\Users\user\Desktop\MitigLLM\models\mitigllm-mistral-sft"
cd C:\Users\user\Desktop\MitigLLM\backend\chatbot
python manage.py runserver
```